# FinSentLLM: QLoRA Fine-Tuning on Kaggle T4 (16GB)

**Model:** Qwen/Qwen2.5-7B-Instruct  
**Dataset:** FinGPT/fingpt-sentiment-train (~76K samples)  
**Eval:** takala/financial_phrasebank sentences_allagree  
**Method:** QLoRA (4-bit NF4 + LoRA r=16)

### Before running:
1. Enable GPU: Notebook Settings → Accelerator → GPU T4 x1
2. Enable Internet: Notebook Settings → Internet → On
3. Add secrets: `WANDB_API_KEY`, `HF_TOKEN` (Kaggle Secrets → Add)


In [ ]:
# Install dependencies (Kaggle has torch pre-installed)
!pip install -q transformers>=4.46.0 datasets peft trl bitsandbytes accelerate wandb huggingface_hub scikit-learn

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

# Login to Hugging Face Hub (needed to push the fine-tuned model)
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen25-finsent-qlora"
HUB_MODEL_ID = ""  # e.g. "your-hf-username/qwen25-7b-finsent-qlora"
PUSH_TO_HUB = bool(HUB_MODEL_ID)

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MAX_SEQ_LENGTH = 512
NUM_EPOCHS = 3
BATCH_SIZE = 2          # T4 16GB: use 2, grad_acc=8 → effective batch 16
GRAD_ACC = 8
LEARNING_RATE = 2e-4
SEED = 42

## 2. Dataset Preparation

In [ ]:
from datasets import load_dataset, DatasetDict

SYSTEM_PROMPT = (
    "You are a financial analyst. Classify the sentiment of the following financial news "
    "sentence from an investor's perspective. Respond with only one word: positive, negative, or neutral."
)
VALID_LABELS = {"positive", "negative", "neutral"}
LABEL_NORMALISATION = {
    "mildly positive": "positive", "mildly negative": "negative",
    "strong positive": "positive", "strong negative": "negative",
    "moderately positive": "positive", "moderately negative": "negative",
}

def format_row(row):
    label = LABEL_NORMALISATION.get(row["output"].strip().lower(), row["output"].strip().lower())
    text = (
        f"<|im_start|>system\n{row.get('instruction', SYSTEM_PROMPT)}<|im_end|>\n"
        f"<|im_start|>user\n{row['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{label}<|im_end|>"
    )
    return {"text": text, "label": label}

raw = load_dataset("FinGPT/fingpt-sentiment-train", split="train")
ds = raw.map(format_row, remove_columns=raw.column_names)
ds = ds.filter(lambda r: r["label"] in VALID_LABELS)

split = ds.train_test_split(test_size=0.05, seed=SEED)
dataset = DatasetDict({"train": split["train"], "validation": split["test"]})
print(dataset)
print(dataset["train"][0]["text"])

## 3. Load Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Use float32 compute dtype — avoids fp16 overflow without a GradScaler
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Train

In [ ]:
import wandb
from pathlib import Path
from trl import SFTConfig, SFTTrainer

wandb.init(project="finsent-qlora", name="qwen25-7b-kaggle-t4", resume="allow")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    max_length=MAX_SEQ_LENGTH,
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    run_name="qwen25-7b-kaggle-t4",
    # Push manually after training completes — mid-training hub pushes cause
    # "no files modified" git errors that corrupt checkpoint tracking.
    push_to_hub=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
)

# Resume from latest checkpoint if one exists (handles Kaggle session drops)
checkpoints = sorted(Path(OUTPUT_DIR).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
resume_from = str(checkpoints[-1]) if checkpoints else None
if resume_from:
    print(f"Resuming from checkpoint: {resume_from}")
else:
    print("No checkpoint found — starting from scratch")

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
if PUSH_TO_HUB:
    trainer.push_to_hub()
wandb.finish()
print(f"Model saved to {OUTPUT_DIR}")

## 5. Evaluation on Financial PhraseBank

In [ ]:
from transformers import pipeline as hf_pipeline
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix
import pandas as pd

FPB_LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

fpb = load_dataset("takala/financial_phrasebank", "sentences_allagree")["train"]
fpb_split = fpb.train_test_split(test_size=0.2, seed=SEED)
test_set = fpb_split["test"]

def parse_label(text):
    text = text.strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in text:
            return label
    return "neutral"

def eval_model(model_path, label):
    pipe = hf_pipeline(
        "text-generation",
        model=model_path,
        tokenizer=tokenizer,
        max_new_tokens=10,
        do_sample=False,
        device_map="auto",
        pad_token_id=tokenizer.eos_token_id,
    )
    preds, true_labels = [], []
    for row in test_set:
        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{row['sentence']}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        out = pipe(prompt)[0]["generated_text"][len(prompt):]
        preds.append(parse_label(out))
        true_labels.append(FPB_LABEL_MAP[row["label"]])

    metrics = {
        "model": label,
        "accuracy": round(accuracy_score(true_labels, preds), 4),
        "weighted_f1": round(f1_score(true_labels, preds, average="weighted", zero_division=0), 4),
        "mcc": round(matthews_corrcoef(true_labels, preds), 4),
    }
    return metrics, true_labels, preds

In [ ]:
# Baseline: base model
baseline_metrics, base_labels, base_preds = eval_model(MODEL_ID, "Qwen2.5-7B-Instruct (zero-shot)")
print(baseline_metrics)

In [ ]:
# Fine-tuned model
ft_metrics, ft_labels, ft_preds = eval_model(OUTPUT_DIR, "Qwen2.5-7B FinSent QLoRA")
print(ft_metrics)

In [ ]:
# Summary table
results_df = pd.DataFrame([baseline_metrics, ft_metrics])
print(results_df.to_markdown(index=False))

import json, pathlib
pathlib.Path("/kaggle/working/results").mkdir(exist_ok=True)
with open("/kaggle/working/results/baseline_results.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)
with open("/kaggle/working/results/finetuned_results.json", "w") as f:
    json.dump(ft_metrics, f, indent=2)

In [ ]:
# Confusion matrices
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

classes = ["negative", "neutral", "positive"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, preds, title in [
    (axes[0], base_preds, "Base Model (zero-shot)"),
    (axes[1], ft_preds, "Fine-tuned QLoRA"),
]:
    cm = confusion_matrix(ft_labels, preds, labels=classes)
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes, ax=ax, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.savefig("/kaggle/working/results/confusion_matrices.png", dpi=150)
plt.show()